# 04 — Run gpt-oss-20b with MXFP4 (OpenAI's Open Model)

**gpt-oss-20b** is OpenAI's open-weight model that ships **natively in MXFP4** format — a 4-bit microscaling float that the T4 can run without additional quantization. At ~13 GB, it fits on the free T4 tier.

OpenAI provides an [official Colab notebook](https://github.com/openai/gpt-oss); this notebook adapts that approach for our `llama-cpp-python` setup.

| Metric | Value |
|--------|-------|
| File size | ~13 GB |
| Quant | MXFP4 (native) |
| Status | ✅ Proven to fit on T4 |

## 1. Install dependencies

In [ ]:
%%capture
pip install llama-cpp-python huggingface_hub --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
pip install transformers accelerate


## 2. Download gpt-oss-20b GGUF

In [ ]:
from huggingface_hub import hf_hub_download, list_repo_files
import os

repo_id = "openai/gpt-oss-20b"  # Hugging Face repo

# Check for GGUF files in the repo
files = list_repo_files(repo_id)
gguf_files = [f for f in files if f.endswith('.gguf')]
print("GGUF files available:")
for f in gguf_files:
    print(f"  {f}")

# If GGUF is available, download it
if gguf_files:
    filename = gguf_files[0]
    model_path = hf_hub_download(repo_id=repo_id, filename=filename)
    print(f"\nDownloaded: {model_path}")
    print(f"Size: {os.path.getsize(model_path) / 1024**3:.2f} GB")
else:
    print("No GGUF found — will use transformers/MXFP4 path below")
    model_path = None


## 3. Option A — Run via llama-cpp-python (if GGUF available)

If a GGUF build of gpt-oss-20b is available, use the standard `llama_cpp.Llama` path:

In [ ]:
from llama_cpp import Llama
import time

if model_path:
    t0 = time.time()
    llm = Llama(
        model_path=model_path,
        n_gpu_layers=-1,
        n_ctx=8192,
        verbose=False,
    )
    print(f"Loaded in {time.time()-t0:.1f}s")
else:
    print("Skipping — no GGUF path. Use Option B below.")


## 4. Option B — Run via transformers (native MXFP4)

If no GGUF is available, load gpt-oss-20b directly via `transformers` using its native MXFP4 weights. This is the path from OpenAI's official Colab notebook.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "openai/gpt-oss-20b"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,  # MXFP4 is handled internally
    device_map="auto",
    trust_remote_code=True,
)
print("Model loaded ✅")
print(f"VRAM: {torch.cuda.memory_allocated()/1024**3:.2f} GB")


In [ ]:
# Generate
messages = [{"role": "user", "content": "What are three benefits of renewable energy?"}]
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(inputs, max_new_tokens=200, do_sample=True, temperature=0.7)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


## Note on MXFP4

MXFP4 (Microscaling FP4) is a native 4-bit floating-point format defined in the OCP Microscaling Formats spec. Unlike GGUF Q4 quants (which pack 4-bit integers), MXFP4 uses true 4-bit floats with a shared exponent per block. gpt-oss-20b ships in this format natively, so no re-quantization is needed.

For the official OpenAI Colab experience, see: https://github.com/openai/gpt-oss